# Silver – DIM_MEDICATION

Goal: Create a standardized medication dimension table containing unique medication information for analytics and joins with fact tables.

Input: capstone.bronze.inventory_raw
Output: capstone.silver.dim_medication

Key steps: select → clean/standardize → generate stable medication IDs → quality checks → deduplicate → write


In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_tbl = "capstone.bronze.inventory_raw"
silver_tbl = "capstone.silver.dim_medication"

df = spark.table(bronze_tbl)

# -------------------------
# 1) Select only what we need (and only if exists)
# -------------------------
keep = [
    "drugName",
    "category",
    "consumeType",
    "manufacturer",
    "description",
    "ingestion_timestamp",
    "source_system"
]
df = df.select([c for c in keep if c in df.columns])

# -------------------------
# 2) Basic cleaning (trim, empty->NULL)
# -------------------------
def clean_str(col):
    return F.when(F.col(col).isNull(), None) \
            .when(F.trim(F.col(col)) == "", None) \
            .otherwise(F.trim(F.col(col)))

df = (df
    .withColumn("medication_name", clean_str("drugName"))
    .withColumn("active_ingredient", clean_str("category"))      # mapping nga tabela jote
    .withColumn("dosage_form", clean_str("consumeType"))         # mapping nga tabela jote
    .withColumn("manufacturer", clean_str("manufacturer"))
    .withColumn("short_description", clean_str("description"))
)

# -------------------------
# 3) Generate stable IDs (hash-based surrogate key)
#    - stable across reruns (unlike monotonically_increasing_id)
# -------------------------
id_key = F.concat_ws(
    "|",
    F.coalesce(F.col("medication_name"), F.lit("")),
    F.coalesce(F.col("active_ingredient"), F.lit("")),
    F.coalesce(F.col("dosage_form"), F.lit("")),
    F.coalesce(F.col("manufacturer"), F.lit(""))
)
df = df.withColumn("medication_id", F.sha2(id_key, 256))

# -------------------------
# 4) Deduplicate (keep latest by ingestion_timestamp)
# -------------------------
w = Window.partitionBy("medication_id").orderBy(F.col("ingestion_timestamp").desc_nulls_last())

df = (df
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# -------------------------
# 5) Final columns (as per diagram)
# -------------------------
silver_df = df.select(
    "medication_id",
    "medication_name",
    "active_ingredient",
    "dosage_form",
    "manufacturer",
    "short_description",
    "ingestion_timestamp",
    "source_system"
)

# -------------------------
# 6) Write to Silver (Delta)
# -------------------------
spark.sql("CREATE SCHEMA IF NOT EXISTS capstone.silver")

(silver_df
 .write
 .mode("overwrite")
 .format("delta")
 .saveAsTable(silver_tbl)
)

display(spark.table(silver_tbl).limit(20))

In [0]:
%sql

-- ===============================
-- 1. Total Rows
-- ===============================
SELECT 
'Total Rows' AS check_type,
COUNT(*) AS result
FROM capstone.silver.dim_medication

UNION ALL

-- ===============================
-- 2. Missing Medication ID
-- ===============================
SELECT
'Missing Medication ID',
COUNT(*)
FROM capstone.silver.dim_medication
WHERE medication_id IS NULL

UNION ALL

-- ===============================
-- 3. Duplicate Medication IDs
-- ===============================
SELECT
'Duplicate Medication IDs',
COUNT(*)
FROM (
  SELECT medication_id
  FROM capstone.silver.dim_medication
  GROUP BY medication_id
  HAVING COUNT(*) > 1
)

UNION ALL

-- ===============================
-- 4. Missing Medication Name
-- ===============================
SELECT
'Missing Medication Name',
COUNT(*)
FROM capstone.silver.dim_medication
WHERE medication_name IS NULL

UNION ALL

-- ===============================
-- 5. Medication Name Too Short (<2)
-- ===============================
SELECT
'Medication Name Too Short',
COUNT(*)
FROM capstone.silver.dim_medication
WHERE medication_name IS NOT NULL AND length(trim(medication_name)) < 2

UNION ALL

-- ===============================
-- 6. Missing Source System
-- ===============================
SELECT
'Missing Source System',
COUNT(*)
FROM capstone.silver.dim_medication
WHERE source_system IS NULL OR trim(source_system) = ''

UNION ALL

-- ===============================
-- 7. Missing Ingestion Timestamp
-- ===============================
SELECT
'Missing Ingestion Timestamp',
COUNT(*)
FROM capstone.silver.dim_medication
WHERE ingestion_timestamp IS NULL

UNION ALL

-- ===============================
-- 8. Description Too Long (>2000)
-- ===============================
SELECT
'Description Too Long',
COUNT(*)
FROM capstone.silver.dim_medication
WHERE short_description IS NOT NULL AND length(short_description) > 2000

UNION ALL

-- ===============================
-- 9. Missing Manufacturer
-- ===============================
SELECT
'Missing Manufacturer',
COUNT(*)
FROM capstone.silver.dim_medication
WHERE manufacturer IS NULL OR trim(manufacturer) = '';